# 1. MPPI - Standing
- Because control frequency should exceed at least 100Hz, real time control using mujoco python is impossible
- The control values are saved with npy array and then replayed for real time control

#### 0. Load scene

In [1]:
import os
import sys
import numpy as np
import time
import mujoco

sys.path.append(os.path.abspath('../'))
from package.pp_base_mujoco.VIEWER import *
from package.pp_base_mujoco.KINEMATICS import *
from package.pp_base_mujoco.UTILS import *

In [2]:
xml_path = './asset/canine_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)
model.opt.timestep = 0.01 # 1000Hz control

joint_names = [
    'FL_HIP_JOINT', 'FL_THIGH_JOINT', 'FL_KNEE_JOINT',
    'FR_HIP_JOINT', 'FR_THIGH_JOINT', 'FR_KNEE_JOINT',
    'HL_HIP_JOINT', 'HL_THIGH_JOINT', 'HL_KNEE_JOINT',
    'HR_HIP_JOINT', 'HR_THIGH_JOINT', 'HR_KNEE_JOINT']
q_init = np.array([0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6])
apply_qpos_names(model, data, joint_names, q_init)
apply_qpos_freejoint(model, data, p=[0, 0, 0.095], rpy=[0, 0, 0])
mujoco.mj_forward(model, data)

#### 1. Declare MPPI Controller

In [3]:
""" COST FUNCTION MPPI """
target_p = np.array([0, 0, 0.3]) # target body position
target_R = np.eye(3) # target body rotation
def cost_function_standing(model, data):
    # Position cost
    body_p = data.xpos[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "base_link")]
    cost_position = np.linalg.norm(body_p - target_p)
    
    # Rotation cost
    body_R = data.xmat[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "base_link")].reshape(3, 3)
    cost_rotation = np.linalg.norm(body_R - target_R)
    
    # ADDED: Velocity penalty to prevent explosive movements
    cost_velocity = np.linalg.norm(data.qvel) * 0.1 
    
    return cost_position + cost_rotation + cost_velocity

In [4]:
import numpy as np
import mujoco
import time

class MPPICONTROLLER:
    def __init__(self, model, data, cost_function, n_sample=100, horizon=100, 
                 lambda_=1.0, sigma=0.5):
        self.model = model
        self.data = data
        self.cost_function = cost_function
        self.n_sample = n_sample
        self.horizon = horizon
        self.n_ctrl = model.nu
        
        self.sigma = sigma
        self.lambda_ = lambda_
        
        # CORRECT: Control sequence is a 2D array (horizon, n_ctrl)
        self.ctrl = np.zeros((self.horizon, self.n_ctrl)) 
        self.epsilon = np.zeros((self.n_sample, self.horizon, self.n_ctrl))

    def sample_epsilon(self):
        # Sample noise for the whole sequence
        self.epsilon = np.random.normal(0.0, self.sigma, 
                                        size=(self.n_sample, self.horizon, self.n_ctrl))
        return self.epsilon
    
    def calc_simulated_cost(self): 
        qpos_bu, qvel_bu = self.data.qpos.copy(), self.data.qvel.copy()
        costs = np.zeros(self.n_sample)
        
        # Generate all sample trajectories at once using broadcasting
        # self.ctrl is (horizon, n_ctrl), epsilon is (n_sample, horizon, n_ctrl)
        ctrl_sampled = self.ctrl + self.epsilon
        
        for k in range(self.n_sample):
            self.data.qpos[:] = qpos_bu
            self.data.qvel[:] = qvel_bu
            mujoco.mj_forward(self.model, self.data)
            
            cost_single_traj = 0.0
            for t in range(self.horizon):
                # Apply the sampled sequence
                self.data.ctrl[:] = ctrl_sampled[k, t, :]
                mujoco.mj_step(self.model, self.data)
                cost_single_traj += self.cost_function(self.model, self.data)
                
            costs[k] = cost_single_traj
            
        # Reset simulator state
        self.data.qpos[:] = qpos_bu
        self.data.qvel[:] = qvel_bu
        mujoco.mj_forward(self.model, self.data)
        
        self.costs = costs
        return costs
    
    def compute_weights(self):
        rho = self.costs.min()
        etas = np.exp((-1.0 / self.lambda_) * (self.costs - rho))
        self.weights = etas / etas.sum()
        return self.weights

    def return_action(self):
        # 1. Update the entire control sequence using standard MPPI math
        # Formula: U_new = U_old + sum(w_k * epsilon_k)
        weighted_epsilon = (self.weights[:, np.newaxis, np.newaxis] * self.epsilon).sum(axis=0)
        self.ctrl += weighted_epsilon
        
        # 2. Extract the first action to apply to the simulator
        action = self.ctrl[0].copy()
        
        # 3. RECEDING HORIZON: Shift sequence forward
        self.ctrl[:-1] = self.ctrl[1:]
        self.ctrl[-1] = self.ctrl[-2] # Pad the end with the last known action
        
        return action

#### 2. Main loop

In [5]:
full_joint_names = get_joint_names(model,data)
print("full joint names:", full_joint_names)
print("joint types:", model.jnt_type)

joint_names = [
    'FL_HIP_JOINT', 'FL_THIGH_JOINT', 'FL_KNEE_JOINT',
    'FR_HIP_JOINT', 'FR_THIGH_JOINT', 'FR_KNEE_JOINT',
    'HL_HIP_JOINT', 'HL_THIGH_JOINT', 'HL_KNEE_JOINT',
    'HR_HIP_JOINT', 'HR_THIGH_JOINT', 'HR_KNEE_JOINT']
q_init = np.array([0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6])
apply_qpos_names(model, data, joint_names, q_init)
apply_qpos_freejoint(model, data, p=[0, 0, 0.095], rpy=[0, 0, 0])
mujoco.mj_forward(model, data)

full joint names: [None, 'FL_HIP_JOINT', 'FL_THIGH_JOINT', 'FL_KNEE_JOINT', 'FR_HIP_JOINT', 'FR_THIGH_JOINT', 'FR_KNEE_JOINT', 'HL_HIP_JOINT', 'HL_THIGH_JOINT', 'HL_KNEE_JOINT', 'HR_HIP_JOINT', 'HR_THIGH_JOINT', 'HR_KNEE_JOINT']
joint types: [0 3 3 3 3 3 3 3 3 3 3 3 3]


In [6]:
mppi_controller = MPPICONTROLLER(
    model=model,
    data=data,
    cost_function=cost_function_standing,
    n_sample=100,   # 100 is usually the minimum for a simple robot
    horizon=30,     # Reduced horizon! 100 steps in MuJoCo might be too long to predict accurately without divergence
    lambda_=0.1,    # Lambda usually needs to be tuned carefully
    sigma=0.5
)

total_steps = 500
ctrl_list = []

for step in range(total_steps):
    start_time = time.time()
    
    mppi_controller.sample_epsilon()
    costs = mppi_controller.calc_simulated_cost()
    mppi_controller.compute_weights()
    action = mppi_controller.return_action()
    
    # apply action to actual simulator step
    data.ctrl[:] = action
    mujoco.mj_step(model, data)
    ctrl_list.append(action)
    
    duration = time.time() - start_time
    print(f"\r Step: {step}, Cost: {costs.min():.4f}, Duration: {duration:.4f} seconds", end="")

 Step: 170, Cost: 174.8780, Duration: 0.1420 secondsWARNING: Nan, Inf or huge value in QACC at DOF 1. The simulation is unstable. Time = 5138.3000.

 Step: 499, Cost: 170.3761, Duration: 0.1490 seconds

In [7]:
full_joint_names = get_joint_names(model,data)
print("full joint names:", full_joint_names)
print("joint types:", model.jnt_type)

joint_names = [
    'FL_HIP_JOINT', 'FL_THIGH_JOINT', 'FL_KNEE_JOINT',
    'FR_HIP_JOINT', 'FR_THIGH_JOINT', 'FR_KNEE_JOINT',
    'HL_HIP_JOINT', 'HL_THIGH_JOINT', 'HL_KNEE_JOINT',
    'HR_HIP_JOINT', 'HR_THIGH_JOINT', 'HR_KNEE_JOINT']
q_init = np.array([0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6,0, 2.3, -2.6])
apply_qpos_names(model, data, joint_names, q_init)
apply_qpos_freejoint(model, data, p=[0, 0, 0.095], rpy=[0, 0, 0])
mujoco.mj_forward(model, data)

full joint names: [None, 'FL_HIP_JOINT', 'FL_THIGH_JOINT', 'FL_KNEE_JOINT', 'FR_HIP_JOINT', 'FR_THIGH_JOINT', 'FR_KNEE_JOINT', 'HL_HIP_JOINT', 'HL_THIGH_JOINT', 'HL_KNEE_JOINT', 'HR_HIP_JOINT', 'HR_THIGH_JOINT', 'HR_KNEE_JOINT']
joint types: [0 3 3 3 3 3 3 3 3 3 3 3 3]


In [8]:
""" MAIN LOOP - CALCULATION """

viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=True)
viewer.view_contact_forces(show=True)

idx = 0

while viewer.is_alive():
    data.ctrl[:] = ctrl_list[idx]
    if idx >= len(ctrl_list)-1:
        pass
    else:
        idx += 1
    mujoco.mj_step(model, data)
    viewer.render()

# close
viewer.close()
del(viewer)

2026-03-14 02:39:37.190 python[37246:4563313] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
